# Fase 1 (revisión) — Paneles mensuales unificados 2014-2025

Este cuaderno corrige el problema de comparabilidad detectado en la revisión: en la
versión anterior Estados Unidos y China se analizaban en periodos distintos. Aquí se
construye **una única ventana temporal, igual para los dos países**, y se incluyen los
**mismos tres activos en cada uno** (índice bursátil local, oro y Bitcoin).

**Decisiones y por qué:**
- **Periodo 2014-2025**: Bitcoin solo se negocia con cierta normalidad desde 2014, así que
  ese año marca el inicio. El final lo marca la disponibilidad del CSI 300.
- **Frecuencia mensual**: el índice de incertidumbre de China (CNEPU) solo existe mensual,
  de modo que se trabaja en mensual para que ambos países sean comparables.
- **Volatilidad realizada mensual**: para cada activo y mes, desviación típica de los
  rendimientos diarios de ese mes (medida directamente observable).

Salidas (en `data/processed/`): `panel_usa_mensual.csv`, `panel_china_mensual.csv`,
`rendimientos_mensuales.csv` y `rendimientos_diarios.csv`.

In [1]:
import os, numpy as np, pandas as pd
ROOT = "/Users/luisgomez/Desktop/kraken/tfg-uncertainty"
RAW  = os.path.join(ROOT, "data/raw")
PROC = os.path.join(ROOT, "data/processed"); os.makedirs(PROC, exist_ok=True)
START, END = "2014-01-01", "2025-07-31"   # final comun: el CSI 300 disponible termina en julio de 2025
print("Periodo objetivo (mismo para todos los activos):", START, "->", END)

Periodo objetivo (mismo para todos los activos): 2014-01-01 -> 2025-07-31


## 1. Carga y limpieza de los precios

Las cotizaciones de oro, Bitcoin y CSI 300 vienen de Investing.com (formato español, coma
decimal). El S&P 500 se actualizó con Yahoo Finance (`sp500-daily-yahoo.csv`) porque el
fichero anterior terminaba en noviembre de 2023. Los índices de incertidumbre proceden de
www.policyuncertainty.com (Baker, Bloom y Davis, 2016; y Davis, Liu y Sheng, 2019).

In [2]:
def num_es(s):   # '4.411,55' -> 4411.55
    return np.nan if pd.isna(s) else float(str(s).replace(".", "").replace(",", "."))
def num_us(s):   # '4,152.02' -> 4152.02
    return np.nan if pd.isna(s) else float(str(s).replace(",", ""))

def load_invertir(path):  # oro, btc (Investing, español)
    df = pd.read_csv(path)
    df["date"]  = pd.to_datetime(df["Fecha"], dayfirst=True, format="%d.%m.%Y", errors="coerce")
    df["close"] = df["Último"].map(num_es)
    return df[["date", "close"]].dropna().sort_values("date").reset_index(drop=True)

def load_yahoo(path):     # S&P 500 (Yahoo: Date,Close)
    df = pd.read_csv(path)
    df["date"]  = pd.to_datetime(df["Date"], errors="coerce")
    df["close"] = pd.to_numeric(df["Close"], errors="coerce")
    return df[["date", "close"]].dropna().sort_values("date").reset_index(drop=True)

def load_csi(path):       # CSI 300 (Investing US: Date MM/DD/YYYY, Price)
    df = pd.read_csv(path)
    df["date"]  = pd.to_datetime(df["Date"], format="%m/%d/%Y", errors="coerce")
    df["close"] = df["Price"].map(num_us)
    return df[["date", "close"]].dropna().sort_values("date").reset_index(drop=True)

sp   = load_yahoo(f"{RAW}/sp500-daily-yahoo.csv")
gold = load_invertir(f"{RAW}/gold-daily.csv")
btc  = load_invertir(f"{RAW}/btc-daily.csv")
csi  = load_csi(f"{RAW}/csi300-daily.csv")

epu = pd.read_csv(f"{RAW}/us-epu-daily.csv")
epu["date"] = pd.to_datetime(dict(year=epu.year, month=epu.month, day=epu.day), errors="coerce")
epu = epu.rename(columns={"daily_policy_index": "EPU"})[["date", "EPU"]].dropna()

cn = pd.read_csv(f"{RAW}/cepu-china-mainland.csv")
cn["date"] = pd.to_datetime(cn["Date"], errors="coerce")
cn = cn.rename(columns={"EPU": "CNEPU"})[["date", "CNEPU"]].dropna()

print("S&P500:", sp.date.min().date(), "->", sp.date.max().date())
print("CSI300:", csi.date.min().date(), "->", csi.date.max().date())

S&P500: 2013-12-31 -> 2025-09-30
CSI300: 2005-01-04 -> 2025-07-29


## 2. Rendimientos y volatilidad realizada

Rendimiento logarítmico diario: R_t = ln(P_t / P_{t-1}).
Volatilidad realizada mensual: desviación típica de los R_t del mes (se exige un mínimo de
12 días de cotización por mes para que el cálculo sea fiable, lo que descarta meses
incompletos).

In [3]:
ASSETS = {"SP500": sp, "CSI300": csi, "Gold": gold, "BTC": btc}

def daily_logret(df):
    d = df[(df.date >= START) & (df.date <= END)].copy()
    d["r"] = np.log(d.close / d.close.shift(1))
    return d.dropna(subset=["r"])

def monthly_vol(df, min_days=12):
    d = daily_logret(df); d["ym"] = d.date.values.astype("datetime64[M]")
    g = d.groupby("ym")["r"]; vol = g.std(ddof=1); n = g.size()
    return vol[n >= min_days]

def monthly_ret(df):
    d = df[(df.date >= START) & (df.date <= END)].copy(); d["ym"] = d.date.values.astype("datetime64[M]")
    last = d.groupby("ym")["close"].last()
    return np.log(last / last.shift(1)).dropna()

vol  = {k: monthly_vol(v) for k, v in ASSETS.items()}
mret = {k: monthly_ret(v) for k, v in ASSETS.items()}

epu_m   = epu.set_index("date")["EPU"].resample("MS").mean()
epu_m   = epu_m[(epu_m.index >= START) & (epu_m.index <= END)]
cnepu_m = cn.set_index("date")["CNEPU"]
cnepu_m = cnepu_m[(cnepu_m.index >= START) & (cnepu_m.index <= END)]

## 3. Construcción de los paneles y guardado

Se arma un panel mensual por país y se restringen ambos a los **meses comunes**, de modo
que la comparación use exactamente la misma muestra.

In [4]:
def panel(epu_s, name, assets):
    df = pd.DataFrame({name: epu_s})
    for a in assets: df[f"vol_{a}"] = vol[a]
    return df.dropna()

usa   = panel(epu_m,   "EPU",   ["SP500", "Gold", "BTC"])
china = panel(cnepu_m, "CNEPU", ["CSI300", "Gold", "BTC"])
common = usa.index.intersection(china.index)
usa, china = usa.loc[common], china.loc[common]

usa.to_csv(f"{PROC}/panel_usa_mensual.csv")
china.to_csv(f"{PROC}/panel_china_mensual.csv")
pd.DataFrame(mret).to_csv(f"{PROC}/rendimientos_mensuales.csv")
pd.DataFrame({a: daily_logret(ASSETS[a]).set_index("date")["r"] for a in ASSETS}).to_csv(f"{PROC}/rendimientos_diarios.csv")

print("Periodo común:", common.min().date(), "->", common.max().date(), "| meses:", len(common))
usa.head()

Periodo común: 2014-01-01 -> 2025-07-01 | meses: 139


,EPU,vol_SP500,vol_Gold,vol_BTC
date,,,,
2014-01-01,95.627419,0.007940,0.008596,0.046066
2014-02-01,83.491429,0.008046,0.007335,0.394315
2014-03-01,72.930645,0.006573,0.009786,0.056470
2014-04-01,63.760000,0.008279,0.007765,0.057344
2014-05-01,68.402581,0.005169,0.008041,0.031856
